# CAM Gallery & Class Prototypes

            这个 notebook 用于人工审阅：

            - 每组正确/错误样本 overlay 图册
            - 每个类别的平均 CAM prototype
            - correct-only 与 wrong-only 的平均 CAM 对照


In [ ]:

from pathlib import Path
import json
import math
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

EXP_ROOT = Path("/data/zengqiang/experiments/ncfm_medmnist_ablation_20260519")

def require_exp_root():
    if not EXP_ROOT.exists():
        raise FileNotFoundError(
            f"EXP_ROOT not found: {EXP_ROOT}. "
            "Edit EXP_ROOT in the first code cell to your experiment directory."
        )

def ensure_report_dir(*parts):
    path = EXP_ROOT / "reports" / "cam" / Path(*parts)
    path.mkdir(parents=True, exist_ok=True)
    return path

def read_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def normalize_group(name):
    if name == "real_train":
        return "real_train"
    if name.startswith("ipc10_"):
        return name[len("ipc10_"):]
    return name

def resolve_result_path(value):
    p = Path(str(value))
    if p.exists():
        return p
    if str(value).startswith("/"):
        return p
    q = EXP_ROOT / value
    return q

def load_eval_metrics():
    require_exp_root()
    rows = []
    for path in sorted((EXP_ROOT / "runs").glob("*/ipc10/*/eval_metrics_best.json")):
        item = read_json(path)
        item["dataset"] = path.parents[2].name
        item["group"] = path.parent.name
        item["metrics_path"] = str(path)
        rows.append(item)
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows)
    order = {"A_pure_ncfd_wopsi": 0, "B_minmax_ncfm_psi": 1, "C_code_default_enhanced": 2}
    df["_order"] = df["group"].map(order).fillna(99)
    return df.sort_values(["dataset", "_order"]).drop(columns=["_order"])

def load_cam_summaries():
    require_exp_root()
    rows = []
    for path in sorted((EXP_ROOT / "results" / "cam").glob("*/*/summary.csv")):
        dataset = path.parents[1].name
        group = normalize_group(path.parent.name)
        df = pd.read_csv(path)
        if df.empty:
            continue
        df["dataset"] = dataset
        df["group"] = group
        df["summary_path"] = str(path)
        rows.append(df)
    if not rows:
        return pd.DataFrame()
    out = pd.concat(rows, ignore_index=True)
    for col in ["index", "y_true", "y_pred", "confidence", "correct", "cam_entropy", "topk_activation_ratio"]:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    return out

def cam_group_summary(cam_df):
    if cam_df.empty:
        return cam_df
    grouped = (
        cam_df.groupby(["dataset", "group"], as_index=False)
        .agg(
            n=("index", "count"),
            cam_acc=("correct", "mean"),
            mean_confidence=("confidence", "mean"),
            mean_entropy=("cam_entropy", "mean"),
            mean_top10_mass=("topk_activation_ratio", "mean"),
            correct_entropy=("cam_entropy", lambda s: s[cam_df.loc[s.index, "correct"] == 1].mean()),
            wrong_entropy=("cam_entropy", lambda s: s[cam_df.loc[s.index, "correct"] == 0].mean()),
            correct_top10_mass=("topk_activation_ratio", lambda s: s[cam_df.loc[s.index, "correct"] == 1].mean()),
            wrong_top10_mass=("topk_activation_ratio", lambda s: s[cam_df.loc[s.index, "correct"] == 0].mean()),
        )
    )
    order = {"real_train": 0, "A_pure_ncfd_wopsi": 1, "B_minmax_ncfm_psi": 2, "C_code_default_enhanced": 3}
    grouped["_order"] = grouped["group"].map(order).fillna(99)
    return grouped.sort_values(["dataset", "_order"]).drop(columns=["_order"])


In [ ]:

cam_df = load_cam_summaries()
if cam_df.empty:
    raise RuntimeError("No CAM summaries found.")
display(cam_group_summary(cam_df))


In [ ]:

def show_overlay_grid(dataset, group, correct=None, n=12, seed=0):
    sub = cam_df[(cam_df.dataset == dataset) & (cam_df.group == group)].copy()
    if correct is not None:
        sub = sub[sub.correct == int(correct)]
    if sub.empty:
        print("No rows for", dataset, group, correct)
        return
    sub = sub.sample(min(n, len(sub)), random_state=seed)
    cols = 4
    rows = math.ceil(len(sub) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes = np.array(axes).reshape(-1)
    for ax, (_, row) in zip(axes, sub.iterrows()):
        path = resolve_result_path(row["overlay_path"])
        if path.exists():
            ax.imshow(Image.open(path))
        ax.set_title(f"idx={int(row['index'])} y={int(row['y_true'])} pred={int(row['y_pred'])} c={row['confidence']:.2f}")
        ax.axis("off")
    for ax in axes[len(sub):]:
        ax.axis("off")
    plt.suptitle(f"{dataset} / {group} / correct={correct}")
    plt.tight_layout()
    plt.show()

# Examples:
# show_overlay_grid("bloodmnist", "real_train", correct=1)
# show_overlay_grid("bloodmnist", "A_pure_ncfd_wopsi", correct=0)


In [ ]:

def read_cam(path_value):
    path = resolve_result_path(path_value)
    if not path.exists():
        return None
    img = np.asarray(Image.open(path).convert("RGB")).astype(np.float32) / 255.0
    cam = img[..., 0]
    cam = cam - cam.min()
    cam = cam / (cam.max() + 1e-12)
    return cam

def build_prototype_table(dataset, group, correct=None):
    sub = cam_df[(cam_df.dataset == dataset) & (cam_df.group == group)].copy()
    if correct is not None:
        sub = sub[sub.correct == int(correct)]
    rows = []
    for cls, cdf in sub.groupby("y_true"):
        cams = []
        for path in cdf["cam_path"]:
            cam = read_cam(path)
            if cam is not None:
                cams.append(cam)
        if cams:
            rows.append((int(cls), np.mean(cams, axis=0), len(cams)))
    return rows

def show_class_prototypes(dataset, group, correct=None):
    protos = build_prototype_table(dataset, group, correct=correct)
    if not protos:
        print("No prototypes for", dataset, group)
        return
    cols = min(5, len(protos))
    rows = math.ceil(len(protos) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
    axes = np.array(axes).reshape(-1)
    for ax, (cls, cam, n) in zip(axes, protos):
        im = ax.imshow(cam, cmap="inferno", vmin=0, vmax=1)
        ax.set_title(f"class {cls}, n={n}")
        ax.axis("off")
    for ax in axes[len(protos):]:
        ax.axis("off")
    plt.suptitle(f"Prototype CAM: {dataset} / {group} / correct={correct}")
    plt.tight_layout()
    plt.show()

# Examples:
# show_class_prototypes("pathmnist", "real_train", correct=1)
# show_class_prototypes("pathmnist", "A_pure_ncfd_wopsi", correct=1)


## 审阅建议

            - 先看 `real_train` 的 class prototype，建立“真实模型长期看哪里”的参考。
            - 再看 A/B/C 的 prototype 是否偏背景、偏边缘、偏固定角落。
            - 错误样本单独看，通常最容易暴露伪特征。
